# Factor Analysis
Compute all four factor types and inspect their values, percentiles, and current zone.

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import date
import pandas as pd

from trading_engine.data.yfinance_loader import YFinanceLoader
from trading_engine.factors.distance_from_peak import DistanceFromPeak
from trading_engine.factors.moving_average import MovingAverageRatio
from trading_engine.factors.bollinger import BollingerBands
from trading_engine.factors.donchian import DonchianChannel
from trading_engine import analyze_factor

loader = YFinanceLoader()

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SYMBOL = "MSFT"
START  = date(2000, 1, 1)
END    = date.today()

prices = loader.load(SYMBOL, START, END)
close  = prices.data["close"]

print(f"Loaded {len(close)} bars for {SYMBOL}")
print(f"Range: {close.index[0].date()} → {close.index[-1].date()}")
print(f"Current close: ${close.iloc[-1]:.4f}")

## 1. Distance From Peak
`value = close / rolling_max(window) - 1`

Range: (-1, 0]. Zero means AT the peak, -0.15 means 15% below.

In [ ]:
WINDOW = 200   # ← change this

factor = DistanceFromPeak(window=WINDOW)
series = factor.compute(prices)
result = analyze_factor(series)

rolling_max = close.rolling(WINDOW).max()
current_peak = rolling_max.iloc[-1]
peak_date = close[close == current_peak].index

print(f"Factor:          {result.factor_name}")
print(f"Current close:   ${close.iloc[-1]:.4f} ({close.index[-1].date()})")
print(f"{WINDOW}-day peak:   ${current_peak:.4f} ({peak_date[-1].date() if len(peak_date) else 'within window'})")
print(f"Factor value:    {result.current_value:.6f}  ({result.current_value*100:.2f}% below peak)")
print(f"Percentile:      {result.current_percentile:.2f}th  (lower = more rare/cheap)")
print(f"History:         {result.history_length_days} trading days")
print()
print("Percentile thresholds (factor value at each percentile):")
for p, v in sorted(result.percentiles.items()):
    marker = " ◀ current" if result.percentiles[p] <= result.current_value < (result.percentiles.get(p+5, 0)) else ""
    print(f"  P{p:2d}: {v:+.4f}  ({v*100:.2f}%){marker}")

In [ ]:
# Inspect the raw factor series
print("Last 10 values:")
print(series.values.tail(10).to_string())

print(f"\nMin ever: {series.values.min():.4f} ({series.values.idxmin().date()})")
print(f"Max ever: {series.values.max():.4f} ({series.values.idxmax().date()})")

## 2. Moving Average Ratio
`value = close / MA(length) - 1`

Positive = price above MA (extended), negative = price below MA (compressed).

In [ ]:
MA_LENGTH = 50   # ← change this
MA_TYPE   = "SMA"  # SMA | EMA | WMA

factor = MovingAverageRatio(ma_type=MA_TYPE, length=MA_LENGTH)
series = factor.compute(prices)
result = analyze_factor(series)

print(f"Factor:        {result.factor_name}")
print(f"Current value: {result.current_value:+.6f}  ({result.current_value*100:+.2f}% from MA)")
print(f"Percentile:    {result.current_percentile:.2f}th")
print()
print("Thresholds:")
for p, v in sorted(result.percentiles.items()):
    print(f"  P{p:2d}: {v:+.4f}")

## 3. Bollinger Band Position
`value = (close - lower_band) / (upper_band - lower_band)`

Range ≈ [0, 1]. Below 0 = below lower band, above 1 = above upper band.

In [ ]:
BB_PERIOD  = 20   # ← change this
BB_STD_DEV = 2.0  # ← change this

factor = BollingerBands(period=BB_PERIOD, num_std=BB_STD_DEV)
series = factor.compute(prices)
result = analyze_factor(series)

print(f"Factor:        {result.factor_name}")
print(f"Current value: {result.current_value:.4f}  (0=lower band, 1=upper band)")
print(f"Percentile:    {result.current_percentile:.2f}th")
print()
print("Thresholds:")
for p, v in sorted(result.percentiles.items()):
    print(f"  P{p:2d}: {v:.4f}")

## 4. Donchian Channel Position
`value = (close - lower) / (upper - lower)` over entry/exit windows.

Near 1.0 = near top of channel (breakout), near 0.0 = near bottom.

In [ ]:
DC_ENTRY  = 20  # ← change this
DC_EXIT   = 10  # typically half of entry

factor = DonchianChannel(entry_length=DC_ENTRY, exit_length=DC_EXIT)
series = factor.compute(prices)
result = analyze_factor(series)

print(f"Factor:        {result.factor_name}")
print(f"Current value: {result.current_value:.4f}")
print(f"Percentile:    {result.current_percentile:.2f}th")
print()
print("Thresholds:")
for p, v in sorted(result.percentiles.items()):
    print(f"  P{p:2d}: {v:.4f}")

## Compare all factors at once

In [ ]:
factors_to_run = [
    ("DistFromPeak(200)",  DistanceFromPeak(window=200)),
    ("DistFromPeak(50)",   DistanceFromPeak(window=50)),
    ("MA Ratio SMA(50)",   MovingAverageRatio(ma_type="SMA", length=50)),
    ("MA Ratio EMA(20)",   MovingAverageRatio(ma_type="EMA", length=20)),
    ("Bollinger(20,2)",    BollingerBands(period=20, num_std=2.0)),
    ("Donchian(20/10)",    DonchianChannel(entry_length=20, exit_length=10)),
]

rows = []
for label, factor in factors_to_run:
    try:
        s = factor.compute(prices)
        r = analyze_factor(s)
        rows.append({
            "Factor": label,
            "Current Value": f"{r.current_value:+.4f}",
            "Percentile": f"{r.current_percentile:.1f}th",
            "P10": f"{r.percentiles.get(10, 0):+.4f}",
            "P50": f"{r.percentiles.get(50, 0):+.4f}",
            "P90": f"{r.percentiles.get(90, 0):+.4f}",
        })
    except Exception as e:
        rows.append({"Factor": label, "Current Value": f"ERROR: {e}", "Percentile": "-", "P10": "-", "P50": "-", "P90": "-"})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows).set_index("Factor")